# 05 – Temporal Alignment (17 variables → 24-hour grid)

Converts irregular physiological measurements into a regular hourly time series over the
first 24 hours of each ICU stay: 24 one-hour bins from INTIME; arithmetic mean of repeated
measurements within a bin; within-stay forward- then backward-fill imputation; a binary
observation mask per variable.

**Run after notebook 04** — uses final_cohort_angus.csv, filtered_chartevents_17vars.csv,
filtered_labevents_17vars.csv.

**Produces:** hourly_vitals.csv (used by notebooks 08, 10, 11; also read by 01).

**Method basis:** preprocessing follows Wang et al. (2022) — implausible-value removal,
hourly resampling, mean aggregation, forward/backward-fill imputation — restricted to the
first 24h; vital-sign definitions follow the official MIT-LCP MIMIC-III code.

MIMIC-III data not included (PhysioNet DUA); see README. Patient-row outputs cleared.

In [ ]:
# --- Setup ---
import os
import gc
import numpy as np
import pandas as pd

try:
    from config import DATA_DIR
except ImportError:
    DATA_DIR = os.environ.get("ERP_DATA_DIR", "./data")

def data_path(name):
    return os.path.join(DATA_DIR, name)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# --- File paths (all under DATA_DIR) ---
COHORT_PATH         = data_path("final_cohort_angus.csv")
CHAR_PATH           = data_path("filtered_chartevents_17vars.csv")
LAB_PATH            = data_path("filtered_labevents_17vars.csv")
HOURLY_PARTIAL_PATH = data_path("hourly_vitals_partial.csv")
HOURLY_RAW_PATH     = data_path("hourly_vitals_raw.csv")
HOURLY_VITALS_PATH  = data_path("hourly_vitals.csv")

print("Cohort:", os.path.exists(COHORT_PATH))
print("CHARTEVENTS:", os.path.exists(CHAR_PATH))
print("LABEVENTS:", os.path.exists(LAB_PATH))

## 1. Load ICU admission times

Loads ICUSTAY_ID and INTIME from the Angus cohort (the authoritative cohort definition).
Basic checks: ICUSTAY_ID numeric, INTIME parsed to datetime, invalid rows dropped, one
admission time per ICU stay.

In [ ]:
# ------------------------------------------------------------------
# Load the Angus cohort
# ------------------------------------------------------------------

cohort = pd.read_csv(
    COHORT_PATH,
    low_memory=False
)

print("Original cohort shape:", cohort.shape)
print("Available columns:")
print(cohort.columns.tolist())

Original cohort shape: (33560, 24)
Available columns:
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'DBSOURCE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'FIRST_WARDID', 'LAST_WARDID', 'INTIME', 'OUTTIME', 'LOS', 'Sepsis_Angus', 'AKI', 'GENDER', 'DOB', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE', 'ETHNICITY', 'HOSPITAL_EXPIRE_FLAG', 'HAS_CHARTEVENTS_DATA', 'AGE']


In [ ]:
# Standardise column names
cohort.columns = (
    cohort.columns
    .str.strip()
    .str.upper()
)

required_cohort_columns = {
    "ICUSTAY_ID",
    "INTIME"
}

missing_columns = required_cohort_columns - set(cohort.columns)

if missing_columns:
    raise ValueError(
        f"Missing required cohort columns: {missing_columns}"
    )

# Convert data types
cohort["ICUSTAY_ID"] = pd.to_numeric(
    cohort["ICUSTAY_ID"],
    errors="coerce"
)

cohort["INTIME"] = pd.to_datetime(
    cohort["INTIME"],
    errors="coerce"
)

# Keep only valid ICU stay IDs and admission times
icu_times = (
    cohort[["ICUSTAY_ID", "INTIME"]]
    .dropna(subset=["ICUSTAY_ID", "INTIME"])
    .copy()
)

icu_times["ICUSTAY_ID"] = (
    icu_times["ICUSTAY_ID"]
    .astype("int64")
)

# Check whether an ICU stay has more than one INTIME
duplicate_intimes = (
    icu_times
    .groupby("ICUSTAY_ID")["INTIME"]
    .nunique()
)

conflicting_intimes = duplicate_intimes[
    duplicate_intimes > 1
]

if len(conflicting_intimes) > 0:
    raise ValueError(
        f"{len(conflicting_intimes):,} ICU stays have conflicting INTIME values."
    )

# Keep one row per ICU stay
icu_times = (
    icu_times
    .drop_duplicates(subset=["ICUSTAY_ID"])
    .sort_values("ICUSTAY_ID")
    .reset_index(drop=True)
)

print(f"ICU stays with valid INTIME: {len(icu_times):,}")
print(
    "Duplicated ICUSTAY_ID:",
    icu_times["ICUSTAY_ID"].duplicated().sum()
)

display(icu_times.head())

ICU stays with valid INTIME: 33,560
Duplicated ICUSTAY_ID: 0


,ICUSTAY_ID,INTIME
0,200003,2199-08-02 19:50:04
1,200007,2109-02-17 10:03:37
2,200009,2189-11-30 10:34:32
3,200014,2105-02-16 23:16:48
4,200019,2178-07-08 09:03:12


## 2. Align vital-sign measurements relative to ICU admission

The filtered vital-sign file (~27M rows) is processed in chunks to stay within memory.
For each observation, hours-from-INTIME = (CHARTTIME − INTIME) / 3600; measurements with
0 ≤ hours < 24 are kept and assigned to an integer hourly bin ICU_HOUR (0–23).

## 3. First-pass hourly aggregation

Measurements are grouped by (ICUSTAY_ID, ICU_HOUR, VARIABLE). Because rows for the same
group can span different file chunks, the sum and count are accumulated (not a per-chunk
mean) so the final hourly mean = total_sum / total_count. A per-chunk "mean of means"
would be wrong when chunk groups have different sizes.

In [ ]:
CHUNK_SIZE = 500_000

usecols = [
    "ICUSTAY_ID",
    "CHARTTIME",
    "VARIABLE",
    "VALUENUM",
    "VALUEUOM"
]

if os.path.exists(HOURLY_PARTIAL_PATH):
    os.remove(HOURLY_PARTIAL_PATH)

icu_intime_map = (
    icu_times
    .set_index("ICUSTAY_ID")["INTIME"]
    .to_dict()
)

header_written = False
source_rows = 0
first_24h_rows = 0
partial_group_rows = 0


def process_file(path, source_name):
    global header_written
    global source_rows
    global first_24h_rows
    global partial_group_rows

    print(f"\nProcessing {source_name}")

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            path,
            usecols=usecols,
            chunksize=CHUNK_SIZE,
            low_memory=False
        ),
        start=1
    ):

        source_rows += len(chunk)

        chunk = chunk.copy()

        # --------------------------------------------------------
        # Basic cleaning
        # --------------------------------------------------------
        chunk["ICUSTAY_ID"] = pd.to_numeric(
            chunk["ICUSTAY_ID"],
            errors="coerce"
        )

        chunk["VALUENUM"] = pd.to_numeric(
            chunk["VALUENUM"],
            errors="coerce"
        )

        chunk["CHARTTIME"] = pd.to_datetime(
            chunk["CHARTTIME"],
            errors="coerce"
        )

        chunk = chunk.dropna(
            subset=[
                "ICUSTAY_ID",
                "CHARTTIME",
                "VARIABLE",
                "VALUENUM"
            ]
        ).copy()

        chunk["ICUSTAY_ID"] = chunk["ICUSTAY_ID"].astype("int64")

        # Keep cohort ICU stays only
        chunk = chunk[
            chunk["ICUSTAY_ID"].isin(icu_intime_map)
        ].copy()

        if chunk.empty:
            continue

        # --------------------------------------------------------
        # Align to ICU admission and keep first 24 h
        # --------------------------------------------------------
        chunk["INTIME"] = chunk["ICUSTAY_ID"].map(
            icu_intime_map
        )

        hours_from_intime = (
            chunk["CHARTTIME"] - chunk["INTIME"]
        ).dt.total_seconds() / 3600

        chunk = chunk[
            (hours_from_intime >= 0)
            & (hours_from_intime < 24)
        ].copy()

        if chunk.empty:
            continue

        hours_from_intime = (
            chunk["CHARTTIME"] - chunk["INTIME"]
        ).dt.total_seconds() / 3600

        chunk["ICU_HOUR"] = np.floor(
            hours_from_intime
        ).astype("int8")

        first_24h_rows += len(chunk)

        # ========================================================
        # Unit standardisation BEFORE hourly aggregation
        # ========================================================

        chunk["VALUEUOM"] = (
            chunk["VALUEUOM"]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )

        # --------------------------------------------------------
        # Temperature -> Celsius
        # --------------------------------------------------------
        temp_mask = chunk["VARIABLE"] == "Temperature"

        fahrenheit_mask = (
            temp_mask
            & (
                chunk["VALUEUOM"].str.contains("f")
                | (chunk["VALUENUM"] > 60)
            )
        )

        chunk.loc[fahrenheit_mask, "VALUENUM"] = (
            chunk.loc[fahrenheit_mask, "VALUENUM"] - 32
        ) * 5 / 9

        # --------------------------------------------------------
        # FiO2 -> fraction (0.21–1.00)
        # --------------------------------------------------------
        fio2_mask = (
            chunk["VARIABLE"]
            == "Fraction inspired oxygen"
        )

        percent_fio2 = (
            fio2_mask
            & (chunk["VALUENUM"] > 1)
        )

        chunk.loc[percent_fio2, "VALUENUM"] = (
            chunk.loc[percent_fio2, "VALUENUM"] / 100
        )

        # --------------------------------------------------------
        # Weight -> kg
        # --------------------------------------------------------
        weight_mask = chunk["VARIABLE"] == "Weight"

        lb_mask = (
            weight_mask
            & chunk["VALUEUOM"].str.contains("lb")
        )

        chunk.loc[lb_mask, "VALUENUM"] = (
            chunk.loc[lb_mask, "VALUENUM"] * 0.45359237
        )

        oz_mask = (
            weight_mask
            & chunk["VALUEUOM"].str.contains("oz")
        )

        chunk.loc[oz_mask, "VALUENUM"] = (
            chunk.loc[oz_mask, "VALUENUM"] * 0.0283495
        )

        # --------------------------------------------------------
        # Height -> cm
        # --------------------------------------------------------
        height_mask = chunk["VARIABLE"] == "Height"

        inch_mask = (
            height_mask
            & chunk["VALUEUOM"].str.contains("inch")
        )

        chunk.loc[inch_mask, "VALUENUM"] = (
            chunk.loc[inch_mask, "VALUENUM"] * 2.54
        )

        # ========================================================
        # Hourly aggregation
        # ========================================================
        grouped = (
            chunk
            .groupby(
                [
                    "ICUSTAY_ID",
                    "ICU_HOUR",
                    "VARIABLE"
                ]
            )["VALUENUM"]
            .agg(
                VALUE_SUM="sum",
                VALUE_COUNT="count"
            )
            .reset_index()
        )

        partial_group_rows += len(grouped)

        grouped.to_csv(
            HOURLY_PARTIAL_PATH,
            mode="a",
            header=not header_written,
            index=False
        )

        header_written = True

        if chunk_number % 20 == 0:
            print(
                f"{source_name} chunk {chunk_number} | "
                f"first-24h rows {first_24h_rows:,}"
            )


process_file(
    CHAR_PATH,
    "CHARTEVENTS"
)

process_file(
    LAB_PATH,
    "LABEVENTS"
)

print("\nFinished first-pass aggregation")
print("Source rows:", f"{source_rows:,}")
print("First 24h rows:", f"{first_24h_rows:,}")
print("Partial groups:", f"{partial_group_rows:,}")



Processing CHARTEVENTS
CHARTEVENTS chunk 20 | first-24h rows 2,650,971
CHARTEVENTS chunk 40 | first-24h rows 5,049,792
CHARTEVENTS chunk 60 | first-24h rows 7,411,861

Processing LABEVENTS

Finished first-pass aggregation
Source rows: 35,273,554
First 24h rows: 8,626,234
Partial groups: 5,884,363


## 4. Second-pass aggregation

Partial sums and counts from all chunks are combined per (ICUSTAY_ID, ICU_HOUR, VARIABLE),
and the final hourly mean is computed as total_sum / total_count.

In [ ]:
# ------------------------------------------------------------------
# Second-pass aggregation across all chunks
# ------------------------------------------------------------------

if not os.path.exists(HOURLY_PARTIAL_PATH):
    raise FileNotFoundError(
        f"Partial aggregation file not found: {HOURLY_PARTIAL_PATH}"
    )

partial = pd.read_csv(
    HOURLY_PARTIAL_PATH,
    low_memory=False
)

print("Partial file shape:", partial.shape)

partial["ICUSTAY_ID"] = pd.to_numeric(
    partial["ICUSTAY_ID"],
    errors="coerce"
)

partial["ICU_HOUR"] = pd.to_numeric(
    partial["ICU_HOUR"],
    errors="coerce"
)

partial["VALUE_SUM"] = pd.to_numeric(
    partial["VALUE_SUM"],
    errors="coerce"
)

partial["VALUE_COUNT"] = pd.to_numeric(
    partial["VALUE_COUNT"],
    errors="coerce"
)

partial = partial.dropna(
    subset=[
        "ICUSTAY_ID",
        "ICU_HOUR",
        "VARIABLE",
        "VALUE_SUM",
        "VALUE_COUNT"
    ]
)

partial["ICUSTAY_ID"] = (
    partial["ICUSTAY_ID"]
    .astype("int64")
)

partial["ICU_HOUR"] = (
    partial["ICU_HOUR"]
    .astype("int8")
)

hourly_long = (
    partial
    .groupby(
        [
            "ICUSTAY_ID",
            "ICU_HOUR",
            "VARIABLE"
        ],
        as_index=False,
        observed=True
    )
    .agg(
        VALUE_SUM=("VALUE_SUM", "sum"),
        VALUE_COUNT=("VALUE_COUNT", "sum")
    )
)

hourly_long["VALUE"] = (
    hourly_long["VALUE_SUM"] /
    hourly_long["VALUE_COUNT"]
)

hourly_long = hourly_long[
    [
        "ICUSTAY_ID",
        "ICU_HOUR",
        "VARIABLE",
        "VALUE",
        "VALUE_COUNT"
    ]
].sort_values(
    [
        "ICUSTAY_ID",
        "ICU_HOUR",
        "VARIABLE"
    ]
).reset_index(drop=True)

print("Final observed hourly groups:", f"{len(hourly_long):,}")
print("Unique ICU stays:", f"{hourly_long['ICUSTAY_ID'].nunique():,}")
print(
    "Hour range:",
    hourly_long["ICU_HOUR"].min(),
    "to",
    hourly_long["ICU_HOUR"].max()
)

display(hourly_long.head(10))

## 5. Convert to a wide hourly representation

The long-format aggregates (one row per stay-hour-variable) are pivoted to wide format
(one row per stay-hour, one column per variable). A complete 24-hour grid is built for
every ICU stay so that stays with entirely missing hours are still represented.

In [ ]:
# ------------------------------------------------------------------
# Define the expected physiological variables
# ------------------------------------------------------------------

EXPECTED_VARIABLES = [
    "heart_rate",
    "sbp",
    "dbp",
    "map",
    "resp_rate",
    "temperature",
    "spo2",
    "fio2",
    "glucose",
    "ph",
    "gcs_eye",
    "gcs_motor",
    "gcs_total",
    "gcs_verbal",
    "weight",
    "height",
]

In [ ]:
# Rename the 17 benchmark variables to model-friendly names
RENAME_MAP = {
    "Heart Rate": "heart_rate",
    "Systolic blood pressure": "sbp",
    "Diastolic blood pressure": "dbp",
    "Mean blood pressure": "map",
    "Respiratory rate": "resp_rate",
    "Temperature": "temperature",
    "Oxygen saturation": "spo2",

    "Fraction inspired oxygen": "fio2",
    "Glucose": "glucose",
    "pH": "ph",

    "Glascow coma scale eye opening": "gcs_eye",
    "Glascow coma scale motor response": "gcs_motor",
    "Glascow coma scale total": "gcs_total",
    "Glascow coma scale verbal response": "gcs_verbal",


    "Weight": "weight",
    "Height": "height",
}

hourly_long["VARIABLE"] = hourly_long["VARIABLE"].replace(RENAME_MAP)

In [ ]:

observed_variables = sorted(
    hourly_long["VARIABLE"]
    .dropna()
    .unique()
    .tolist()
)

print("Observed variables:", observed_variables)

unexpected_variables = sorted(
    set(observed_variables) - set(EXPECTED_VARIABLES)
)

missing_expected_variables = sorted(
    set(EXPECTED_VARIABLES) - set(observed_variables)
)

print("Unexpected variables:", unexpected_variables)
print("Expected variables absent from data:", missing_expected_variables)

Observed variables: ['dbp', 'fio2', 'gcs_eye', 'gcs_motor', 'gcs_total', 'gcs_verbal', 'glucose', 'heart_rate', 'height', 'map', 'ph', 'resp_rate', 'sbp', 'spo2', 'temperature', 'weight']
Unexpected variables: []
Expected variables absent from data: []


### Validation

The extracted physiological variables exactly matched the seven predefined
vital-sign variables expected from the official MIT-LCP implementation.

No unexpected variables were detected and no expected variables were missing,
confirming that the previous extraction step produced the intended physiological
measurements.

In [ ]:
# Pivot observed hourly means into wide format
hourly_wide_observed = (
    hourly_long
    .pivot_table(
    index=["ICUSTAY_ID","ICU_HOUR"],
    columns="VARIABLE",
    values="VALUE",
    aggfunc="first"
)
    .reset_index()
)

hourly_wide_observed.columns.name = None

# Ensure that every expected variable exists as a column
for variable in EXPECTED_VARIABLES:
    if variable not in hourly_wide_observed.columns:
        hourly_wide_observed[variable] = np.nan

hourly_wide_observed = hourly_wide_observed[
    [
        "ICUSTAY_ID",
        "ICU_HOUR",
        *EXPECTED_VARIABLES
    ]
]

# Create all combinations of cohort ICU stays and hours 0–23
complete_index = pd.MultiIndex.from_product(
    [
        icu_times["ICUSTAY_ID"].sort_values().unique(),
        np.arange(24, dtype=np.int8)
    ],
    names=[
        "ICUSTAY_ID",
        "ICU_HOUR"
    ]
)

hourly_raw = (
    hourly_wide_observed
    .set_index(
        [
            "ICUSTAY_ID",
            "ICU_HOUR"
        ]
    )
    .reindex(complete_index)
    .reset_index()
)

hourly_raw = hourly_raw.sort_values(
    [
        "ICUSTAY_ID",
        "ICU_HOUR"
    ]
).reset_index(drop=True)

print("Hourly raw shape:", hourly_raw.shape)
print(
    "Expected number of rows:",
    f"{len(icu_times) * 24:,}"
)

display(hourly_raw.head(30))

In [ ]:
print("Hourly dataset summary")
print("-"*40)

print("ICU stays:",
      hourly_raw["ICUSTAY_ID"].nunique())

print("Rows:",
      len(hourly_raw))

print("Hours:",
      hourly_raw["ICU_HOUR"].min(),
      "-",
      hourly_raw["ICU_HOUR"].max())

print("Variables:")
print(EXPECTED_VARIABLES)

print("\nMissing values")

display(
    hourly_raw[EXPECTED_VARIABLES]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

Hourly dataset summary
----------------------------------------
ICU stays: 33560
Rows: 805440
Hours: 0 - 23
Variables:
['heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'temperature', 'spo2', 'fio2', 'glucose', 'ph', 'gcs_eye', 'gcs_motor', 'gcs_total', 'gcs_verbal', 'weight', 'height']

Missing values


,0
heart_rate,9.32
sbp,11.11
dbp,11.13
map,11.49
resp_rate,10.61
temperature,65.09
spo2,11.94
fio2,93.84
glucose,71.06
ph,84.03


In [ ]:
for col in EXPECTED_VARIABLES:
    x = hourly_raw[col].dropna()

    print(
        f"{col:12s} "
        f"min={x.min():8.2f}  "
        f"p1={x.quantile(.01):8.2f}  "
        f"median={x.median():8.2f}  "
        f"p99={x.quantile(.99):8.2f}  "
        f"max={x.max():8.2f}"
    )

heart_rate   min=    0.00  p1=   50.00  median=   84.00  p99=  132.33  max=  941.00
sbp          min=    0.00  p1=   79.35  median=  116.75  p99=  177.00  max=119119.02
dbp          min=    0.00  p1=   32.00  median=   59.00  p99=   99.00  max=60127.00
map          min=  -36.00  p1=   50.00  median=   76.33  p99=  119.67  max=120130.03
resp_rate    min=    0.00  p1=    7.33  median=   18.00  p99=   35.00  max=588897.75
temperature  min=  -17.78  p1=   34.72  median=   37.00  p99=   39.00  max=  536.39
spo2         min=    0.00  p1=   85.00  median=   98.00  p99=  100.00  max=981023.00
fio2         min=    0.00  p1=    0.30  median=    0.50  p99=    1.00  max=   10.00
glucose      min=    0.00  p1=   63.00  median=  129.00  p99=  383.00  max=500053.00
ph           min=    0.00  p1=    5.00  median=    7.37  p99=    7.54  max=  750.00
gcs_eye      min=    1.00  p1=    1.00  median=    4.00  p99=    4.00  max=    4.00
gcs_motor    min=    1.00  p1=    1.00  median=    6.00  p99=    6.00  

In [ ]:
# ============================================================
# Clean clinical variables before imputation
# ============================================================

clean = hourly_raw.copy()

# ------------------------------------------------------------
# Remove physiologically implausible values
# Out-of-range values become NaN
# ------------------------------------------------------------

VALID_RANGES = {
    "heart_rate":  (20, 250),
    "sbp":         (40, 300),
    "dbp":         (20, 200),
    "map":         (20, 250),
    "resp_rate":   (1, 80),
    "temperature": (25, 45),
    "spo2":        (20, 100),

    "fio2":        (0.21, 1.00),
    "glucose":     (20, 1000),
    "ph":          (6.5, 8.0),

    "gcs_eye":     (1, 4),
    "gcs_motor":   (1, 6),
    "gcs_total":   (3, 15),
    "gcs_verbal":  (1, 5),

    "weight":      (20, 300),
    "height":      (50, 250),
}

for col, (low, high) in VALID_RANGES.items():

    bad = (
        clean[col].notna()
        & ~clean[col].between(low, high)
    )

    print(
        f"{col:12s}: removing {bad.sum():,} "
        f"({bad.mean()*100:.3f}%)"
    )

    clean.loc[bad, col] = np.nan

heart_rate  : removing 35 (0.004%)
sbp         : removing 252 (0.031%)
dbp         : removing 667 (0.083%)
map         : removing 242 (0.030%)
resp_rate   : removing 1,480 (0.184%)
temperature : removing 315 (0.039%)
spo2        : removing 95 (0.012%)
fio2        : removing 305 (0.038%)
glucose     : removing 43 (0.005%)
ph          : removing 5,584 (0.693%)
gcs_eye     : removing 0 (0.000%)
gcs_motor   : removing 0 (0.000%)
gcs_total   : removing 0 (0.000%)
gcs_verbal  : removing 0 (0.000%)
weight      : removing 317 (0.039%)
height      : removing 7 (0.001%)


In [ ]:
for col in EXPECTED_VARIABLES:
    x = clean[col].dropna()

    print(
        f"{col:12s} "
        f"n={len(x):8d}  "
        f"min={x.min():8.2f}  "
        f"p1={x.quantile(.01):8.2f}  "
        f"median={x.median():8.2f}  "
        f"p99={x.quantile(.99):8.2f}  "
        f"max={x.max():8.2f}"
    )

print("\nMissing after cleaning:")
display(
    clean[EXPECTED_VARIABLES]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

heart_rate   n=  730310  min=   20.00  p1=   50.00  median=   84.00  p99=  132.33  max=  210.00
sbp          n=  715680  min=   40.00  p1=   80.00  median=  117.00  p99=  177.00  max=  274.00
dbp          n=  715138  min=   20.00  p1=   33.00  median=   59.00  p99=   99.00  max=  197.00
map          n=  712624  min=   20.00  p1=   50.00  median=   76.33  p99=  119.50  max=  250.00
resp_rate    n=  718539  min=    1.00  p1=    8.00  median=   18.00  p99=   35.00  max=   80.00
temperature  n=  280847  min=   25.00  p1=   34.80  median=   37.00  p99=   39.00  max=   42.22
spo2         n=  709191  min=   20.00  p1=   85.00  median=   98.00  p99=  100.00  max=  100.00
fio2         n=   49327  min=    0.21  p1=    0.30  median=    0.50  p99=    1.00  max=    1.00
glucose      n=  233027  min=   21.00  p1=   63.00  median=  129.00  p99=  382.00  max= 1000.00
ph           n=  123078  min=    6.50  p1=    6.56  median=    7.37  p99=    7.54  max=    8.00
gcs_eye      n=  156643  min=    1.00  p

,0
heart_rate,9.33
sbp,11.14
dbp,11.21
map,11.52
resp_rate,10.79
temperature,65.13
spo2,11.95
fio2,93.88
glucose,71.07
ph,84.72


## 6. Missingness before imputation

Missingness is quantified before forward/backward filling, distinguishing hourly
missingness (proportion of empty stay-hour cells) from stay-level absence (variables never
measured in the whole 24h window). Filling only works when at least one value exists for
that variable in the stay; variables absent for the entire window stay missing.

In [ ]:
# ------------------------------------------------------------------
# Missingness before imputation
# ------------------------------------------------------------------

hourly_missing_before = (
    clean[EXPECTED_VARIABLES]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent_before")
    .to_frame()
)

stay_level_absence_before = (
    clean
    .groupby("ICUSTAY_ID")[EXPECTED_VARIABLES]
    .count()
    .eq(0)
    .sum()
    .sort_values(ascending=False)
    .rename("icu_stays_with_no_measurement")
    .to_frame()
)

print("Hourly missingness before imputation:")
display(hourly_missing_before)

print("ICU stays with no measurement during the full 24-hour window:")
display(stay_level_absence_before)

Hourly missingness before imputation:


,missing_percent_before
height,99.283125
weight,95.633567
fio2,93.875770
ph,84.719160
gcs_total,80.733760
gcs_motor,80.651445
gcs_verbal,80.608984
gcs_eye,80.551872
glucose,71.068360
temperature,65.131233


ICU stays with no measurement during the full 24-hour window:


,icu_stays_with_no_measurement
height,27790
fio2,25040
gcs_total,14316
gcs_verbal,14312
gcs_motor,14310
gcs_eye,14307
weight,11342
ph,11258
temperature,1019
glucose,465


## 7. Forward filling followed by backward filling

Missing values are imputed within each ICU stay and variable: forward fill (carry the last
observed value forward), then backward fill (fill leading gaps from the first later
observation). Imputation is per-stay, so values never cross between patients.

In [ ]:
# ------------------------------------------------------------------
# Forward fill followed by backward fill within each ICU stay
# ------------------------------------------------------------------

hourly_imputed = clean.copy()

hourly_imputed[EXPECTED_VARIABLES] = (
    hourly_imputed
    .groupby(
        "ICUSTAY_ID",
        sort=False
    )[EXPECTED_VARIABLES]
    .transform(
        lambda group: group.ffill().bfill()
    )
)

print("Imputation complete.")
display(hourly_imputed.head(30))

In [ ]:
hourly_imputed[EXPECTED_VARIABLES].isna().mean()

,0
heart_rate,0.012426
sbp,0.012485
dbp,0.012485
map,0.012574
resp_rate,0.013498
temperature,0.030364
spo2,0.010250
fio2,0.746126
glucose,0.013856
ph,0.335459


## 8. Missingness after imputation

Filling removes missing values wherever a stay-variable has at least one observation in the
24h window; variables never observed remain missing. These are not imputed with
dataset-wide statistics here — that is deferred to the modelling notebook and fitted on the
training set only, to avoid leakage.

In [ ]:
# ------------------------------------------------------------------
# Missingness after imputation
# ------------------------------------------------------------------

hourly_missing_after = (
    hourly_imputed[EXPECTED_VARIABLES]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent_after")
    .to_frame()
)

missingness_comparison = (
    hourly_missing_before
    .join(hourly_missing_after)
)

missingness_comparison["percentage_point_reduction"] = (
    missingness_comparison["missing_percent_before"] -
    missingness_comparison["missing_percent_after"]
)

print("Missingness before and after imputation:")
display(missingness_comparison)

Missingness before and after imputation:


,missing_percent_before,missing_percent_after,percentage_point_reduction
height,99.283125,82.806913,16.476212
weight,95.633567,33.796186,61.837381
fio2,93.875770,74.612634,19.263136
ph,84.719160,33.545888,51.173272
gcs_total,80.733760,42.657926,38.075834
gcs_motor,80.651445,42.640048,38.011397
gcs_verbal,80.608984,42.646007,37.962977
gcs_eye,80.551872,42.631108,37.920764
glucose,71.068360,1.385578,69.682782
temperature,65.131233,3.036353,62.094880


In [ ]:
# ------------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------------

hours_per_stay = (
    hourly_imputed
    .groupby("ICUSTAY_ID")
    .size()
)

invalid_hour_counts = hours_per_stay[
    hours_per_stay != 24
]

duplicate_stay_hours = hourly_imputed.duplicated(
    subset=[
        "ICUSTAY_ID",
        "ICU_HOUR"
    ]
).sum()

invalid_hour_values = hourly_imputed.loc[
    ~hourly_imputed["ICU_HOUR"].between(0, 23)
]

print("Number of ICU stays:", f"{hourly_imputed['ICUSTAY_ID'].nunique():,}")
print("Total hourly rows:", f"{len(hourly_imputed):,}")
print("Expected hourly rows:", f"{len(icu_times) * 24:,}")
print("ICU stays not containing exactly 24 rows:", len(invalid_hour_counts))
print("Duplicated ICU stay-hour rows:", duplicate_stay_hours)
print("Rows with invalid ICU_HOUR:", len(invalid_hour_values))

assert hourly_imputed["ICUSTAY_ID"].nunique() == len(icu_times)
assert len(hourly_imputed) == len(icu_times) * 24
assert len(invalid_hour_counts) == 0
assert duplicate_stay_hours == 0
assert len(invalid_hour_values) == 0

print("\nStructural validation passed.")

Number of ICU stays: 33,560
Total hourly rows: 805,440
Expected hourly rows: 805,440
ICU stays not containing exactly 24 rows: 0
Duplicated ICU stay-hour rows: 0
Rows with invalid ICU_HOUR: 0

Structural validation passed.


## 9. Physiological plausibility validation

Values were filtered to the MIT-LCP physiological ranges before aggregation; averaging and
forward/backward filling cannot produce out-of-range values, so the aligned data remain
within the same valid ranges.

In [ ]:
# ------------------------------------------------------------------
# Physiological / clinical range validation
# ------------------------------------------------------------------

VALID_RANGES = {
    "heart_rate":  (20, 250),
    "sbp":         (40, 300),
    "dbp":         (20, 200),
    "map":         (20, 250),
    "resp_rate":   (1, 80),
    "temperature": (25, 45),
    "spo2":        (20, 100),

    "fio2":        (0.21, 1.00),
    "glucose":     (20, 1000),
    "ph":          (6.5, 8.0),

    "gcs_eye":     (1, 4),
    "gcs_motor":   (1, 6),
    "gcs_total":   (3, 15),
    "gcs_verbal":  (1, 5),

    "weight":      (20, 300),
    "height":      (50, 250),
}

range_validation = []

for variable, (lower, upper) in VALID_RANGES.items():

    series = hourly_imputed[variable]

    below_range = int((series < lower).sum())
    above_range = int((series > upper).sum())
    non_missing = int(series.notna().sum())

    range_validation.append(
        {
            "variable": variable,
            "valid_lower_bound": lower,
            "valid_upper_bound": upper,
            "non_missing_values": non_missing,
            "below_range": below_range,
            "above_range": above_range
        }
    )

range_validation = pd.DataFrame(range_validation)

display(range_validation)

assert range_validation["below_range"].sum() == 0
assert range_validation["above_range"].sum() == 0

print("Clinical range validation passed.")

,variable,valid_lower_bound,valid_upper_bound,non_missing_values,below_range,above_range
0,heart_rate,20.00,250.0,795432,0,0
1,sbp,40.00,300.0,795384,0,0
2,dbp,20.00,200.0,795384,0,0
3,map,20.00,250.0,795312,0,0
4,resp_rate,1.00,80.0,794568,0,0
5,temperature,25.00,45.0,780984,0,0
6,spo2,20.00,100.0,797184,0,0
7,fio2,0.21,1.0,204480,0,0
8,glucose,20.00,1000.0,794280,0,0
9,ph,6.50,8.0,535248,0,0


Clinical range validation passed.


## 10. Create observation masks

For each variable, a binary mask records whether the value was directly observed (1) or
imputed (0) at each hour. Masks preserve measurement-pattern information without changing
the imputed values — relevant in ICU data because measurement frequency itself can reflect
patient condition. Both values and masks are used as model inputs.

In [ ]:
# ------------------------------------------------------------------
# Create binary observation masks BEFORE imputation
# ------------------------------------------------------------------

for variable in EXPECTED_VARIABLES:

    hourly_imputed[f"{variable}_observed"] = (
        clean[variable]
        .notna()
        .astype("int8")
    )

mask_columns = [
    f"{variable}_observed"
    for variable in EXPECTED_VARIABLES
]

print("Mask columns:")
print(mask_columns)

print("Number of masks:", len(mask_columns))

display(
    hourly_imputed[
        [
            "ICUSTAY_ID",
            "ICU_HOUR",
            *EXPECTED_VARIABLES,
            *mask_columns
        ]
    ].head(10)
)

### 11. Save the temporally aligned datasets

- `hourly_vitals_raw.csv` — hourly means before imputation (kept for reproducibility and
  missingness analysis).
- `hourly_vitals.csv` — final aligned dataset: one row per stay-hour, the physiological
  variables (forward/backward-filled) and their binary masks.

In [ ]:
# ------------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------------

clean.to_csv(
    HOURLY_RAW_PATH,
    index=False
)

hourly_imputed.to_csv(
    HOURLY_VITALS_PATH,
    index=False
)

print("Saved cleaned hourly data before imputation:")
print(HOURLY_RAW_PATH)

print("\nSaved temporally imputed hourly data:")
print(HOURLY_VITALS_PATH)

print("\nClean hourly shape:", clean.shape)
print("Final hourly shape:", hourly_imputed.shape)

In [ ]:
hv = pd.read_csv(HOURLY_VITALS_PATH)
print(hv.columns.tolist())
print(hv.shape)
hv.head()

['ICUSTAY_ID', 'ICU_HOUR', 'heart_rate', 'sbp', 'dbp', 'map', 'resp_rate', 'temperature', 'spo2', 'fio2', 'glucose', 'ph', 'gcs_eye', 'gcs_motor', 'gcs_total', 'gcs_verbal', 'weight', 'height', 'heart_rate_observed', 'sbp_observed', 'dbp_observed', 'map_observed', 'resp_rate_observed', 'temperature_observed', 'spo2_observed', 'fio2_observed', 'glucose_observed', 'ph_observed', 'gcs_eye_observed', 'gcs_motor_observed', 'gcs_total_observed', 'gcs_verbal_observed', 'weight_observed', 'height_observed']
(805440, 34)


,ICUSTAY_ID,ICU_HOUR,heart_rate,sbp,dbp,map,resp_rate,temperature,spo2,fio2,glucose,ph,gcs_eye,gcs_motor,gcs_total,gcs_verbal,weight,height,heart_rate_observed,sbp_observed,dbp_observed,map_observed,resp_rate_observed,temperature_observed,spo2_observed,fio2_observed,glucose_observed,ph_observed,gcs_eye_observed,gcs_motor_observed,gcs_total_observed,gcs_verbal_observed,weight_observed,height_observed
0,200003,0,119.0,91.000000,49.000000,58.000000,35.000000,38.999999,97.0,NaN,159.0,7.36,4.0,6.0,15.0,5.0,NaN,NaN,1,1,1,1,1,0,1,0,0,0,1,1,1,1,0,0
1,200003,1,118.0,88.333333,52.000000,59.000000,32.000000,38.999999,96.0,NaN,159.0,7.36,4.0,6.0,15.0,5.0,NaN,NaN,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
2,200003,2,116.0,85.333333,52.333333,59.666667,30.333333,38.277790,95.0,NaN,159.0,7.36,4.0,6.0,15.0,5.0,NaN,NaN,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
3,200003,3,112.0,86.500000,60.500000,65.500000,32.500000,37.777790,93.5,NaN,159.0,7.36,4.0,6.0,15.0,5.0,NaN,NaN,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0
4,200003,4,108.0,89.500000,61.000000,67.000000,37.000000,36.833318,91.5,NaN,159.0,7.36,4.0,6.0,15.0,5.0,NaN,NaN,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0,0


## 12. Final validation summary

Checks that every cohort stay has exactly 24 hourly rows (indices 0–23) with no duplicates,
all expected variables present, values within physiological ranges, and missingness reduced
after filling (residual missingness only where a variable was absent for the whole window).

In [ ]:
# ------------------------------------------------------------------
# Final validation summary
# ------------------------------------------------------------------

final_summary = pd.DataFrame(
    {
        "metric": [
            "Angus cohort ICU stays",
            "ICU stays in hourly dataset",
            "Hourly rows",
            "Expected hourly rows",
            "Hours per ICU stay",
            "Minimum ICU hour",
            "Maximum ICU hour",
            "Duplicated ICU stay-hour rows",
            "Number of physiological variables",
            "Number of observation masks"
        ],
        "value": [
            len(icu_times),
            hourly_imputed["ICUSTAY_ID"].nunique(),
            len(hourly_imputed),
            len(icu_times) * 24,
            "24",
            hourly_imputed["ICU_HOUR"].min(),
            hourly_imputed["ICU_HOUR"].max(),
            hourly_imputed.duplicated(
                ["ICUSTAY_ID", "ICU_HOUR"]
            ).sum(),
            len(EXPECTED_VARIABLES),
            len(mask_columns)
        ]
    }
)

display(final_summary)

,metric,value
0,Angus cohort ICU stays,33560
1,ICU stays in hourly dataset,33560
2,Hourly rows,805440
3,Expected hourly rows,805440
4,Hours per ICU stay,24
5,Minimum ICU hour,0
6,Maximum ICU hour,23
7,Duplicated ICU stay-hour rows,0
8,Number of physiological variables,16
9,Number of observation masks,16


In [ ]:
# 1. Temperature unit sanity check
print("TEMP:", hourly_imputed["temperature"].dropna().median())

# 2. Imputation check across the 16 variables
print(hourly_imputed[EXPECTED_VARIABLES].isna().mean().mul(100).round(1).sort_values())



TEMP: 36.888888888888886
spo2            1.0
heart_rate      1.2
dbp             1.2
sbp             1.2
map             1.3
resp_rate       1.3
glucose         1.4
temperature     3.0
ph             33.5
weight         33.8
gcs_motor      42.6
gcs_eye        42.6
gcs_verbal     42.6
gcs_total      42.7
fio2           74.6
height         82.8
dtype: float64
